In [ ]:
# BirdCLEF 2026 — ENB0 SED Ensemble Inference
#
# Postprocessing:
#  1. Texture-aware time smoothing — heavier for Insecta/Amphibia (continuous)
#                                  — standard for Aves/Mammalia/Reptilia (discrete)
#  2. Site/hour priors — Bayesian smoothing from labeled soundscape statistics

import math, re
from pathlib import Path
from time import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
import torchvision
import timm


# ── CONFIG ────────────────────────────────────────────────────────────────────

ROOT     = Path('/kaggle/input/competitions/birdclef-2026')
OUT_PATH = Path('./submission.csv')

CKPT_PATHS = [
    Path('/kaggle/input/notebooks/denizegememetoglu/birdclef-sed/outputs/best_sed_fold0.pth'),
    Path('/kaggle/input/notebooks/aliozanmemetoglu/birdclef-sed-fold-1/outputs/best_sed_fold1.pth'),
    Path('/kaggle/input/notebooks/aliozanmemetoglu/birdclef-sed-fold-2/outputs/best_sed_fold2.pth'),
    Path('/kaggle/input/models/aliozanmemetoglu/birdclef-sed-fold-3/pytorch/default/1/outputs/best_sed_fold3.pth'),
    Path('/kaggle/input/models/aliozanmemetoglu/birdclef-sed-fold-4/pytorch/default/1/outputs/best_sed_fold4.pth'),
]

# Prior smoothing — uses labeled soundscape statistics
PRIOR_WEIGHT = 0.15   # blend weight for site/hour prior

SR          = 32_000
CLIP_SEC    = 5
CLIP_SAMP   = SR * CLIP_SEC
NUM_SEG     = 12
TTA_SHIFT   = SR * 5 // 4
NUM_CLASSES = 234
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BACKBONE    = 'tf_efficientnetv2_b0'

MEL_CFG = dict(
    n_fft=2048, hop_length=512, n_mels=256,
    f_min=20, f_max=16000,
    mel_scale='slaney', norm='slaney',
    target_size=(256, 256), top_db=80.0,
)

# Time smoothing weights — different for texture vs event taxa
# Texture (Insecta/Amphibia): produce continuous ambient sound
#   → heavier smoothing [0.35, 0.30, 0.35] spreads signal across windows
# Event (Aves/Mammalia/Reptilia): discrete calls
#   → standard [0.20, 0.60, 0.20] preserves temporal precision
SMOOTH_EVENT   = np.array([0.20, 0.60, 0.20])
SMOOTH_TEXTURE = np.array([0.35, 0.30, 0.35])

# ── TAXONOMY ──────────────────────────────────────────────────────────────────

def load_taxonomy(label_cols):
    """Return boolean masks for texture and event classes."""
    tax = pd.read_csv(ROOT / 'taxonomy.csv')
    tax['primary_label'] = tax['primary_label'].astype(str)
    label_to_idx = {l: i for i, l in enumerate(label_cols)}
    texture_classes = set(
        tax[tax['class_name'].isin(['Insecta', 'Amphibia'])]['primary_label']
    )
    texture_mask = np.array([
        label_to_idx.get(l, -1) for l in label_cols
    ])  # just use boolean arrays below
    is_texture = np.array([
        l in texture_classes for l in label_cols
    ], dtype=bool)
    return is_texture


# ── SPECTROGRAM ───────────────────────────────────────────────────────────────

class Spectrogram(nn.Module):
    def __init__(self, n_fft=2048, hop_length=512, n_mels=256,
                 f_min=20, f_max=16000, mel_scale='slaney', norm='slaney',
                 target_size=(256, 256), top_db=80.0, **_):
        super().__init__()
        self.top_db = top_db
        self.mel_transform = T.MelSpectrogram(
            sample_rate=SR, n_fft=n_fft, hop_length=hop_length,
            n_mels=n_mels, f_min=f_min, f_max=f_max,
            mel_scale=mel_scale, norm=norm,
            pad_mode='constant', power=2.0, center=True,
        )
        self.resize = torchvision.transforms.Resize(target_size, antialias=True)

    def power_to_db(self, S):
        S        = S.float()
        amin     = 1e-10
        log_spec = 10.0 * torch.log10(S.clamp(min=amin))
        log_spec -= 10.0 * math.log10(amin)
        if self.top_db is not None:
            max_val  = log_spec.flatten(-2).max(dim=-1).values[..., None, None]
            log_spec = torch.maximum(log_spec, max_val - self.top_db)
        return log_spec

    def forward(self, x):
        mel  = self.mel_transform(x.float())
        mel  = self.power_to_db(mel)
        mel  = mel.unsqueeze(1)
        mel  = self.resize(mel)
        B    = mel.shape[0]
        flat = mel.reshape(B, -1)
        mins = flat.min(dim=1).values[:, None, None, None]
        maxs = flat.max(dim=1).values[:, None, None, None]
        return (mel - mins) / (maxs - mins + 1e-7)


# ── SED MODEL ─────────────────────────────────────────────────────────────────

class SEDModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = timm.create_model(
            BACKBONE, pretrained=False,
            num_classes=0, global_pool='',
            in_chans=1, drop_rate=0.0, drop_path_rate=0.0,
        )
        self.dropout = nn.Dropout(0.5)
        self.fc      = nn.Linear(self.backbone.num_features, num_classes)

    def forward(self, x):
        feat = self.backbone(x).mean(dim=2)
        feat = self.dropout(feat).permute(0, 2, 1)
        return self.fc(feat).max(dim=1).values


# ── LOAD CNN MODELS ───────────────────────────────────────────────────────────

def load_models():
    models, label2idx = [], None
    for p in CKPT_PATHS:
        if not Path(p).exists():
            print(f'  SKIP (not found): {p}')
            continue
        ckpt = torch.load(str(p), map_location=DEVICE, weights_only=False)
        if label2idx is None:
            label2idx = ckpt['label2idx']
        m = SEDModel(len(label2idx)).to(DEVICE)
        m.load_state_dict(ckpt['model_state'])
        m.eval()
        models.append(m)
        print(f'  Loaded: epoch={ckpt["epoch"]}  auc={ckpt["auc"]:.4f}  {Path(p).name}')
    if not models:
        raise RuntimeError('No models loaded — check CKPT_PATHS')
    print(f'  Ensemble: {len(models)} folds')
    return models, label2idx


# ── SITE / HOUR PRIORS ────────────────────────────────────────────────────────

# ── SITE / HOUR PRIORS ────────────────────────────────────────────────────────

def build_prior_tables(label_cols):
    """
    Build per-site and per-hour species probability tables from labeled soundscapes.
    Uses Bayesian smoothing toward global prior.
    """
    sc  = pd.read_csv(ROOT / 'train_soundscapes_labels.csv')
    sc  = sc.drop_duplicates(subset=['filename', 'start', 'end'])
    tax = pd.read_csv(ROOT / 'taxonomy.csv')
    label_to_idx = {l: i for i, l in enumerate(label_cols)}
    n  = len(label_cols)

    def parse_meta(filename):
        parts = filename.split('_')
        site  = parts[3] if len(parts) > 3 else 'UNK'
        hour  = int(parts[5][:2]) if len(parts) > 5 else -1
        return site, hour

    # Build multi-hot label matrix
    rows = []
    for _, r in sc.iterrows():
        site, hour = parse_meta(r['filename'])
        y = np.zeros(n, dtype=np.float32)
        for sp in str(r['primary_label']).split(';'):
            sp = sp.strip()
            if sp in label_to_idx:
                y[label_to_idx[sp]] = 1.0
        rows.append((site, hour, y))

    global_p = np.mean([r[2] for r in rows], axis=0).astype(np.float32)

    # Per-site
    sites = {}
    for site, hour, y in rows:
        if site not in sites:
            sites[site] = []
        sites[site].append(y)
    site_p = {s: np.mean(v, axis=0).astype(np.float32) for s, v in sites.items()}
    site_n = {s: len(v) for s, v in sites.items()}

    # Per-hour
    hours = {}
    for site, hour, y in rows:
        if hour < 0:
            continue
        if hour not in hours:
            hours[hour] = []
        hours[hour].append(y)
    hour_p = {h: np.mean(v, axis=0).astype(np.float32) for h, v in hours.items()}
    hour_n = {h: len(v) for h, v in hours.items()}

    print(f'  Prior tables: {len(site_p)} sites, {len(hour_p)} hours, '
          f'global_p mean={global_p.mean():.4f}')
    return {
        'global_p': global_p,
        'site_p':   site_p,
        'site_n':   site_n,
        'hour_p':   hour_p,
        'hour_n':   hour_n,
    }


def get_prior(site, hour, tables, smooth_k=8.0):
    """
    Bayesian-smoothed prior for a given site+hour.
    p_final = w_site * p_site + (1-w_site) * global_p
    then blend with hour prior similarly.
    smooth_k: pseudo-count for smoothing (higher = more shrinkage toward global)
    """
    p = tables['global_p'].copy()

    if hour >= 0 and hour in tables['hour_p']:
        nh = tables['hour_n'][hour]
        wh = nh / (nh + smooth_k)
        p  = wh * tables['hour_p'][hour] + (1 - wh) * p

    if site in tables['site_p']:
        ns = tables['site_n'][site]
        ws = ns / (ns + smooth_k)
        p  = ws * tables['site_p'][site] + (1 - ws) * p

    return p.astype(np.float32)


def parse_filename_meta(stem):
    """Extract site and hour from soundscape filename stem."""
    # BC2026_Test_0001_S05_20250227_010002
    parts = stem.split('_')
    site  = parts[3] if len(parts) > 3 else 'UNK'
    hour  = int(parts[5][:2]) if len(parts) > 5 else -1
    return site, hour


# ── TEXTURE-AWARE TIME SMOOTHING ──────────────────────────────────────────────

def time_smooth(preds, is_texture):
    """
    Apply different smoothing kernels to texture vs event classes.
    preds: (12, n_classes)
    is_texture: (n_classes,) bool mask
    """
    if NUM_SEG <= 2:
        return preds

    def smooth(p, w):
        pad = np.pad(p, ((1, 1), (0, 0)), mode='edge')
        return w[0] * pad[:-2] + w[1] * pad[1:-1] + w[2] * pad[2:]

    result = preds.copy()
    if is_texture.any():
        result[:, is_texture]  = smooth(preds[:, is_texture],  SMOOTH_TEXTURE)
    if (~is_texture).any():
        result[:, ~is_texture] = smooth(preds[:, ~is_texture], SMOOTH_EVENT)
    return result


# ── AUDIO ─────────────────────────────────────────────────────────────────────

def load_segments(filepath, offset_samp=0):
    try:
        wav, sr = torchaudio.load(str(filepath))
        wav = wav.mean(0).float()
        if sr != SR:
            wav = torchaudio.functional.resample(wav, sr, SR)
        target = SR * 60
        if wav.shape[0] < target:
            pad = torch.zeros(target)
            pad[:wav.shape[0]] = wav
            wav = pad
        else:
            wav = wav[:target]
        if offset_samp > 0:
            wav = torch.roll(wav, offset_samp)
        return wav.reshape(NUM_SEG, CLIP_SAMP)
    except Exception as e:
        print(f'  Warning: {Path(filepath).name}: {e}')
        return torch.zeros(NUM_SEG, CLIP_SAMP)


# ── INFERENCE ─────────────────────────────────────────────────────────────────

def run_inference():
    print(f'Device: {DEVICE}')

    print('Loading CNN fold models...')
    models, label2idx = load_models()
    label_cols = [l for l, _ in sorted(label2idx.items(), key=lambda x: x[1])]
    n_classes  = len(label_cols)

    print('Loading taxonomy...')
    is_texture = load_taxonomy(label_cols)
    print(f'  Texture classes: {is_texture.sum()}  Event classes: {(~is_texture).sum()}')

    mel = Spectrogram(**MEL_CFG).to(DEVICE)
    mel.eval()

    # Site/hour priors
    print('Building site/hour prior tables...')
    prior_tables = build_prior_tables(label_cols)

    # Test files
    test_dir = ROOT / 'test_soundscapes'
    files    = sorted(test_dir.glob('*.ogg'))
    if len(files) == 0:
        sample = pd.read_csv(ROOT / 'sample_submission.csv')
        stems  = sorted(set('_'.join(r.split('_')[:-1]) for r in sample['row_id']))
        files  = [test_dir / f'{s}.ogg' for s in stems]
        print(f'Test folder empty — dry run on {len(files)} files')
    print(f'Test files: {len(files)}')

    all_row_ids, all_preds = [], []
    t0 = time()

    for i, fpath in enumerate(files):
        stem    = Path(fpath).stem
        row_ids = [f'{stem}_{(j+1)*CLIP_SEC}' for j in range(NUM_SEG)]
        site, hour = parse_filename_meta(stem)

        # CNN inference
        segs       = load_segments(str(fpath), offset_samp=0).to(DEVICE)
        segs_shift = load_segments(str(fpath), offset_samp=TTA_SHIFT).to(DEVICE)

        with torch.no_grad():
            with torch.amp.autocast('cuda', enabled=False):
                specs       = mel(segs)
                specs_shift = mel(segs_shift)

            fold_probs, fold_probs_shift = [], []
            for m in models:
                fold_probs.append(torch.sigmoid(m(specs)).cpu().numpy())
                fold_probs_shift.append(torch.sigmoid(m(specs_shift)).cpu().numpy())

        cnn_preds = (np.mean(fold_probs, axis=0) +
                     np.mean(fold_probs_shift, axis=0)) / 2.0  # (12, n_classes)

        preds = cnn_preds

        # Texture-aware time smoothing
        preds = time_smooth(preds, is_texture)

        # Site/hour prior blending
        prior = get_prior(site, hour, prior_tables)  # (n_classes,)
        # Broadcast prior across 12 windows and blend
        preds = (1 - PRIOR_WEIGHT) * preds + PRIOR_WEIGHT * prior[None, :]

        all_row_ids.extend(row_ids)
        all_preds.append(preds)

        if (i + 1) % 200 == 0 or i == 0:
            elapsed   = time() - t0
            fps       = (i + 1) / elapsed
            remaining = (len(files) - i - 1) / fps if fps > 0 else 0
            print(f'  {i+1}/{len(files)}  {elapsed:.0f}s  '
                  f'~{remaining/60:.1f} min remaining')

    all_preds = np.concatenate(all_preds, axis=0)

    sub = pd.DataFrame(all_preds, columns=label_cols)
    sub.insert(0, 'row_id', all_row_ids)

    sample = pd.read_csv(ROOT / 'sample_submission.csv')
    assert list(sub.columns) == list(sample.columns), \
        f'Column mismatch!\nExpected: {list(sample.columns)[:5]}\nGot: {list(sub.columns)[:5]}'

    sub.to_csv(OUT_PATH, index=False)
    elapsed = time() - t0
    print(f'\nDone in {elapsed/60:.1f} min')
    print(f'Submission: {sub.shape}  '
          f'predictions [{all_preds.min():.4f}, {all_preds.max():.4f}]')
    print(f'Prior blending: ON (w={PRIOR_WEIGHT})')


if __name__ == '__main__':
    run_inference()
    